In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [2]:
%%bash
cd /kaggle/working
if [ ! -d repo/.git ]; then
  git clone https://github.com/Ajha2005/Sequential-Fraud-Detection.git repo
fi
cd repo
git restore reports 2>/dev/null || true
git fetch origin main
git checkout -B main origin/main
git log --oneline -1
grep -q start_day src/sfraud/data/merge.py && echo "OK: start-day card_id code" || { echo "WRONG CODE: start_day missing, stop here"; exit 1; }

branch 'main' set up to track 'origin/main'.
Your branch is up to date with 'origin/main'.
c433778 Add the full project report as a Word document
OK: start-day card_id code


Cloning into 'repo'...
From https://github.com/Ajha2005/Sequential-Fraud-Detection
 * branch            main       -> FETCH_HEAD
Reset branch 'main'


In [3]:
%%bash
cd /kaggle/working/repo
git fetch origin main && git checkout -B main origin/main
git log --oneline -1
rm -rf data/processed/* checkpoints/*

branch 'main' set up to track 'origin/main'.
Your branch is up to date with 'origin/main'.
c433778 Add the full project report as a Word document


From https://github.com/Ajha2005/Sequential-Fraud-Detection
 * branch            main       -> FETCH_HEAD
Reset branch 'main'


In [4]:
%cd /kaggle/working/repo
!pip install -q -r requirements.txt
!pip install -q -e .

/kaggle/working/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sfraud (pyproject.toml) ... done


In [5]:
%%bash
cd /kaggle/working/repo
mkdir -p data/raw
D=/kaggle/input/competitions/ieee-fraud-detection
for f in train_transaction.csv train_identity.csv; do
  test -f "$D/$f" || { echo "MISSING $D/$f: attach the competition under Add Input"; exit 1; }
  ln -sf "$D/$f" "data/raw/$f"
done
ls -lL data/raw

total 693244
-rw-r--r-- 1 nobody nogroup  26529680 Oct  2 01:48 train_identity.csv
-rw-r--r-- 1 nobody nogroup 683351067 Oct  2 01:51 train_transaction.csv


In [6]:
%%bash
cd /kaggle/working/repo
rm -rf data/processed/* checkpoints/*
ls data/processed checkpoints

checkpoints:

data/processed:


In [7]:
import torch
print("cuda:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

cuda: False 


In [8]:
%cd /kaggle/working/repo
!python scripts/phase1_eda.py

/kaggle/working/repo
Loading real IEEE-CIS data from /kaggle/working/repo/data/raw
Saved merged dataset (590540 rows) to /kaggle/working/repo/data/processed/merged.parquet
EDA stats: {
  "n_rows": 590540,
  "n_cards": 161453,
  "fraud_rate": 0.03499000914417313,
  "missing_values": {
    "card_category": 1571,
    "addr1": 65706
  },
  "wide_columns_missing": {
    "n_columns": 414,
    "n_columns_with_missing": 399,
    "mean_missing_fraction": 0.45722830606687825,
    "max_missing_fraction": 0.9919615944728554
  },
  "data_source": "real_kaggle",
  "seq_len_stats": {
    "mean": 3.6576588852483387,
    "median": 1.0,
    "p90": 8.0,
    "p95": 13.0,
    "max": 1414
  }
}


In [9]:
import pandas as pd
m = pd.read_parquet("/kaggle/working/repo/data/processed/merged.parquet", columns=["card_id"])
miss = m.card_id.str.contains("MISSING_")
real = m.card_id[~miss].value_counts()
print("rows with a missing core field (each its own card):", int(miss.sum()))
print("real cards:", len(real))
print(real.describe(percentiles=[.5, .9, .99]))
print("share of rows in the 10 biggest cards: %.3f" % (real.head(10).sum() / len(m)))
print(real.head(10))

rows with a missing core field (each its own card): 12286
real cards: 149167
count    149167.000000
mean          3.876554
std           8.092654
min           1.000000
50%           2.000000
90%           9.000000
99%          32.000000
max        1414.000000
Name: count, dtype: float64
share of rows in the 10 biggest cards: 0.007
card_id
15775_481.0_150.0_102.0_129.0    1414
9500_321.0_150.0_226.0_-85.0      480
7919_194.0_150.0_166.0_92.0       439
7919_194.0_150.0_166.0_124.0      285
7919_194.0_150.0_202.0_34.0       242
8900_385.0_150.0_226.0_-60.0      231
7919_194.0_150.0_202.0_63.0       226
8528_215.0_150.0_226.0_-159.0     215
7919_194.0_150.0_202.0_1.0        212
7919_194.0_150.0_166.0_93.0       206
Name: count, dtype: int64


In [10]:
%%bash
cd /kaggle/working
if [ ! -d repo/.git ]; then
  git clone https://github.com/Ajha2005/Sequential-Fraud-Detection.git repo
fi
cd repo
git restore reports 2>/dev/null || true
git fetch origin main
git checkout -B main origin/main
git log --oneline -1
grep -q start_day src/sfraud/data/merge.py && echo "OK: start-day card_id code" || { echo "WRONG CODE: start_day missing, stop here"; exit 1; }

branch 'main' set up to track 'origin/main'.
Your branch is up to date with 'origin/main'.
c433778 Add the full project report as a Word document
OK: start-day card_id code


From https://github.com/Ajha2005/Sequential-Fraud-Detection
 * branch            main       -> FETCH_HEAD
Reset branch 'main'


In [11]:
%%bash
cd /kaggle/working/repo
git fetch origin main && git checkout -B main origin/main
git log --oneline -1
grep -c "xgboost_plus" configs/config.yaml

branch 'main' set up to track 'origin/main'.
Your branch is up to date with 'origin/main'.
c433778 Add the full project report as a Word document
2


From https://github.com/Ajha2005/Sequential-Fraud-Detection
 * branch            main       -> FETCH_HEAD
Reset branch 'main'


In [12]:
%cd /kaggle/working/repo
!pip install -q -e .
!python -c "import sfraud; print(sfraud.__file__)"

/kaggle/working/repo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sfraud (pyproject.toml) ... done
/kaggle/working/repo/src/sfraud/__init__.py


In [13]:
%cd /kaggle/working/repo
!python scripts/phase1_eda.py

/kaggle/working/repo
EDA stats: {
  "n_rows": 590540,
  "n_cards": 161453,
  "fraud_rate": 0.03499000914417313,
  "missing_values": {
    "card_category": 1571,
    "addr1": 65706
  },
  "wide_columns_missing": {
    "n_columns": 414,
    "n_columns_with_missing": 399,
    "mean_missing_fraction": 0.45722830606687825,
    "max_missing_fraction": 0.9919615944728554
  },
  "data_source": "real_kaggle",
  "seq_len_stats": {
    "mean": 3.6576588852483387,
    "median": 1.0,
    "p90": 8.0,
    "p95": 13.0,
    "max": 1414
  }
}


In [14]:
!pip install -q -r requirements.txt

In [15]:
%%bash
cd /kaggle/working/repo
echo "--- inputs:"; ls /kaggle/input
echo "--- current data/raw:"; ls -lL data/raw 2>&1
D=/kaggle/input/competitions/ieee-fraud-detection
for f in train_transaction.csv train_identity.csv; do
  test -f "$D/$f" || { echo "MISSING $D/$f -- re-attach the competition under Add Input, then rerun"; exit 1; }
  ln -sf "$D/$f" "data/raw/$f"
done
echo "--- after relink:"; ls -lL data/raw

--- inputs:
competitions
--- current data/raw:
total 693244
-rw-r--r-- 1 nobody nogroup  26529680 Oct  2 01:48 train_identity.csv
-rw-r--r-- 1 nobody nogroup 683351067 Oct  2 01:51 train_transaction.csv
--- after relink:
total 693244
-rw-r--r-- 1 nobody nogroup  26529680 Oct  2 01:48 train_identity.csv
-rw-r--r-- 1 nobody nogroup 683351067 Oct  2 01:51 train_transaction.csv


In [16]:
%%bash
cd /kaggle/working/repo
rm -rf data/processed/* checkpoints/*
python scripts/phase1_eda.py

Loading real IEEE-CIS data from /kaggle/working/repo/data/raw
Saved merged dataset (590540 rows) to /kaggle/working/repo/data/processed/merged.parquet
EDA stats: {
  "n_rows": 590540,
  "n_cards": 161453,
  "fraud_rate": 0.03499000914417313,
  "missing_values": {
    "card_category": 1571,
    "addr1": 65706
  },
  "wide_columns_missing": {
    "n_columns": 414,
    "n_columns_with_missing": 399,
    "mean_missing_fraction": 0.45722830606687825,
    "max_missing_fraction": 0.9919615944728554
  },
  "data_source": "real_kaggle",
  "seq_len_stats": {
    "mean": 3.6576588852483387,
    "median": 1.0,
    "p90": 8.0,
    "p95": 13.0,
    "max": 1414
  }
}


In [17]:
%cd /kaggle/working/repo
!ls -lL data/raw
!python -c "import sfraud; print(sfraud.__file__)"

/kaggle/working/repo
total 693244
-rw-r--r-- 1 nobody nogroup  26529680 Oct  2 01:48 train_identity.csv
-rw-r--r-- 1 nobody nogroup 683351067 Oct  2 01:51 train_transaction.csv
/kaggle/working/repo/src/sfraud/__init__.py


In [18]:
%cd /kaggle/working/repo
!python scripts/phase2_build_sequences.py

/kaggle/working/repo
Effective features: 377 numeric (9 base + 368 extension), 15 categorical (6 base + 9 extension)
Saved features_xgb.parquet (590540 rows, 497 features, 20 of them history-derived) to /kaggle/working/repo/data/processed
split=train -> {'n_cards': 113017, 'n_transactions': 425401, 'fraud_rate': 0.030514737591147423}
split=val -> {'n_cards': 24218, 'n_transactions': 69426, 'fraud_rate': 0.04969319701194763}
split=test -> {'n_cards': 24218, 'n_transactions': 60182, 'fraud_rate': 0.04537901654839516}
Saved sequences + feature_meta.json to /kaggle/working/repo/data/processed


In [19]:
%cd /kaggle/working/repo
!python scripts/phase3_train_baseline.py

/kaggle/working/repo
Train rows=455832, fraud=14160, scale_pos_weight=31.19
xgboost  val   AUC-ROC=0.8912  PR-AUC=0.5701  precision=0.225  recall=0.769  acc=0.8560
xgboost  test  AUC-ROC=0.8792  PR-AUC=0.5287  precision=0.184  recall=0.765  acc=0.8322
logreg   val   AUC-ROC=0.8513  PR-AUC=0.2495  precision=0.151  recall=0.800  acc=0.7652
logreg   test  AUC-ROC=0.8452  PR-AUC=0.3961  precision=0.132  recall=0.775  acc=0.7544


In [20]:
!python scripts/phase3b_train_xgb_plus.py

XGBoost-plus: 497 features (20 history-derived, 477 without), device=cpu
search max_depth=6 colsample=0.40 -> val PR-AUC=0.6119 (best_iter=267)
search max_depth=6 colsample=0.70 -> val PR-AUC=0.6013 (best_iter=361)
search max_depth=9 colsample=0.40 -> val PR-AUC=0.6042 (best_iter=129)
search max_depth=9 colsample=0.70 -> val PR-AUC=0.6037 (best_iter=73)
search picked {'max_depth': 6, 'colsample_bytree': 0.4}
xgboost_plus         val   AUC-ROC=0.9069  PR-AUC=0.6104  (best_iter=558)
xgboost_plus         test  AUC-ROC=0.8938  PR-AUC=0.5519  (best_iter=558)
xgboost_plus_nohist  val   AUC-ROC=0.8951  PR-AUC=0.6165  (best_iter=1083)
xgboost_plus_nohist  test  AUC-ROC=0.8926  PR-AUC=0.5606  (best_iter=1083)
History ablation (test PR-AUC, with minus without history features): -0.0087


In [21]:
!python scripts/phase4_train_deep_models.py    

=== Training LSTM ===
LSTM loss=focal, pos_weight=31.77, params=210449
epoch 01/15  train_loss=0.0108  val_auc=0.8619  val_pr_auc=0.4639
epoch 02/15  train_loss=0.0097  val_auc=0.8650  val_pr_auc=0.4745
epoch 03/15  train_loss=0.0095  val_auc=0.8673  val_pr_auc=0.4902
epoch 04/15  train_loss=0.0094  val_auc=0.8654  val_pr_auc=0.4920
epoch 05/15  train_loss=0.0093  val_auc=0.8623  val_pr_auc=0.4683
epoch 06/15  train_loss=0.0093  val_auc=0.8689  val_pr_auc=0.4839
epoch 07/15  train_loss=0.0091  val_auc=0.8714  val_pr_auc=0.4947
epoch 08/15  train_loss=0.0091  val_auc=0.8725  val_pr_auc=0.4962
epoch 09/15  train_loss=0.0090  val_auc=0.8660  val_pr_auc=0.4872
epoch 10/15  train_loss=0.0089  val_auc=0.8593  val_pr_auc=0.4638
epoch 11/15  train_loss=0.0089  val_auc=0.8729  val_pr_auc=0.4907
epoch 12/15  train_loss=0.0088  val_auc=0.8695  val_pr_auc=0.4982
epoch 13/15  train_loss=0.0088  val_auc=0.8705  val_pr_auc=0.5030
epoch 14/15  train_loss=0.0088  val_auc=0.8573  val_pr_auc=0.4879
epoch

In [22]:
!python scripts/phase5_fault_fraud.py

=== Training autoencoder anomaly head ===
Autoencoder training on 441672 normal rows (train), validating on 69579 normal rows
epoch 01/15  train_loss=0.5660  val_loss=148.7256
epoch 02/15  train_loss=0.4239  val_loss=146.9199
epoch 03/15  train_loss=0.4020  val_loss=139.7568
epoch 04/15  train_loss=0.3748  val_loss=124.6106
epoch 05/15  train_loss=0.3634  val_loss=109.2581
epoch 06/15  train_loss=0.3591  val_loss=111.7463
epoch 07/15  train_loss=0.3464  val_loss=108.4210
epoch 08/15  train_loss=0.3423  val_loss=103.5096
epoch 09/15  train_loss=0.3338  val_loss=100.8355
epoch 10/15  train_loss=0.3269  val_loss=104.6854
epoch 11/15  train_loss=0.3297  val_loss=95.3681
epoch 12/15  train_loss=0.3258  val_loss=91.8190
epoch 13/15  train_loss=0.3244  val_loss=103.1249
epoch 14/15  train_loss=0.3272  val_loss=92.7878
epoch 15/15  train_loss=0.3245  val_loss=96.5359
Saved anomaly scores for 590540 transactions to /kaggle/working/repo/data/processed/anomaly_scores.parquet
=== Fault/fraud quadr

In [23]:
import json, os
import numpy as np
import pandas as pd

BASE = os.environ.get("SFRAUD_PROCESSED", "/kaggle/working/repo/data/processed")
OUT = os.environ.get("SFRAUD_OUT", "/kaggle/working")
SEED, PER_BAND = 0, 12

q = pd.read_parquet(f"{BASE}/fault_fraud_quadrants.parquet")
flat = pd.read_parquet(f"{BASE}/features_flat.parquet").set_index("TransactionID")
num = json.load(open(f"{BASE}/feature_meta.json"))["numeric_features"]
ctx = pd.read_parquet(f"{BASE}/merged.parquet", columns=["TransactionID", "ProductCD", "email_domain", "device_type", "card_type", "card_category"]).set_index("TransactionID")

fault = q[q.quadrant == "Fault"].copy()
normal = q[q.quadrant == "Normal"].sample(min(20000, int((q.quadrant == "Normal").sum())), random_state=SEED).copy()

def extremeness(df):
    """Per row: the largest |z| among the (train-standardized) numeric features, and the 3 most extreme features."""
    X = flat.loc[df.TransactionID, num].to_numpy(dtype=np.float32)
    A = np.abs(X)
    top = np.argsort(-A, axis=1)[:, :3]
    names = np.array(num)
    out = pd.DataFrame({"max_abs_z": A.max(axis=1)}, index=df.index)
    for k in range(3):
        out[f"top{k+1}_feature"] = names[top[:, k]]
        out[f"top{k+1}_z"] = np.take_along_axis(X, top[:, [k]], axis=1)[:, 0].round(1)
    return out

fault = fault.join(extremeness(fault)).join(ctx, on="TransactionID")
normal = normal.join(extremeness(normal)).join(ctx, on="TransactionID")

def share(s, t): return float((s > t).mean())
summary = {
    "n_fault_rows": int(len(fault)),
    "n_distinct_cards_in_fault": int(fault.card_id.nunique()),
    "top10_cards_share_of_fault_rows": float(fault.card_id.value_counts().head(10).sum() / len(fault)),
    "fraud_rate_fault": float(fault.isFraud.mean()),
    "fraud_rate_normal_sample": float(normal.isFraud.mean()),
    "max_abs_z": {
        "fault_median": float(fault.max_abs_z.median()), "normal_median": float(normal.max_abs_z.median()),
        "fault_p90": float(fault.max_abs_z.quantile(.9)), "normal_p90": float(normal.max_abs_z.quantile(.9)),
        **{f"share_fault_gt_{t}": share(fault.max_abs_z, t) for t in (5, 10, 50)},
        **{f"share_normal_gt_{t}": share(normal.max_abs_z, t) for t in (5, 10, 50)},
    },
    "amount": {"fault_median": float(fault.TransactionAmt.median()), "normal_median": float(normal.TransactionAmt.median()),
               "fault_p90": float(fault.TransactionAmt.quantile(.9)), "normal_p90": float(normal.TransactionAmt.quantile(.9))},
    "most_common_top_feature_in_fault": fault.top1_feature.value_counts().head(10).to_dict(),
    "ProductCD_share_fault": fault.ProductCD.value_counts(normalize=True).round(3).to_dict(),
    "ProductCD_share_normal": normal.ProductCD.value_counts(normalize=True).round(3).to_dict(),
}

edges = fault.anomaly_score.quantile([0, .5, .8, .95, .99, 1.0]).to_numpy()
labels = ["0-50%", "50-80%", "80-95%", "95-99%", "99-100%"]
fault["band"] = pd.cut(fault.anomaly_score, bins=edges, labels=labels, include_lowest=True)
bands = fault.groupby("band", observed=True).agg(
    rows=("TransactionID", "size"), score_min=("anomaly_score", "min"), score_max=("anomaly_score", "max"),
    fraud_rate=("isFraud", "mean"), median_amount=("TransactionAmt", "median"), median_max_abs_z=("max_abs_z", "median"),
    share_maxz_gt10=("max_abs_z", lambda s: (s > 10).mean()),
).round(3)
summary["bands"] = bands.reset_index().to_dict(orient="records")

parts = [g.sample(min(PER_BAND, len(g)), random_state=SEED) for _, g in fault.groupby("band", observed=True)]
sample = pd.concat(parts).sort_values("anomaly_score", ascending=False)
cols = ["band", "TransactionID", "card_id", "n_seen", "TransactionAmt", "ProductCD", "email_domain", "device_type", "card_type",
        "fraud_prob", "anomaly_score", "isFraud", "max_abs_z", "top1_feature", "top1_z", "top2_feature", "top2_z", "top3_feature", "top3_z"]
sample = sample[cols].copy()
sample["reviewer_label"] = ""   # fill in: benign-looking / suspicious / unclear
sample["reviewer_note"] = ""
sample.to_csv(f"{OUT}/fault_review_sample.csv", index=False)
json.dump(summary, open(f"{OUT}/fault_review_summary.json", "w"), indent=2, default=float)

print(json.dumps({k: v for k, v in summary.items() if k != "bands"}, indent=2, default=float))
print("\nBy anomaly-score band inside the Fault quadrant:\n", bands.to_string())
print(f"\nWrote {len(sample)} sampled rows to {OUT}/fault_review_sample.csv and the summary to {OUT}/fault_review_summary.json")

{
  "n_fault_rows": 11687,
  "n_distinct_cards_in_fault": 6434,
  "top10_cards_share_of_fault_rows": 0.017284161889278685,
  "fraud_rate_fault": 0.061350218191152564,
  "fraud_rate_normal_sample": 0.0251,
  "max_abs_z": {
    "fault_median": 11.804405212402344,
    "normal_median": 3.0633857250213623,
    "fault_p90": 25.075821304321277,
    "normal_p90": 5.9995408058166655,
    "share_fault_gt_5": 0.9648327201163687,
    "share_fault_gt_10": 0.667065970736716,
    "share_fault_gt_50": 0.009668862839051938,
    "share_normal_gt_5": 0.1745,
    "share_normal_gt_10": 0.0111,
    "share_normal_gt_50": 0.0
  },
  "amount": {
    "fault_median": 72.98999786376953,
    "normal_median": 67.94999694824219,
    "fault_p90": 527.0,
    "normal_p90": 303.1878997802775
  },
  "most_common_top_feature_in_fault": {
    "D14": 448,
    "D7": 416,
    "V300": 352,
    "V108": 342,
    "V107": 341,
    "V313": 336,
    "V114": 329,
    "V146": 301,
    "V141": 292,
    "D12": 284
  },
  "ProductCD_shar

In [24]:
!python scripts/phase6_evaluate.py

=== Model comparison + confidence curve ===
xgboost      test AUC-ROC=0.8792  PR-AUC=0.5287
logreg       test AUC-ROC=0.8452  PR-AUC=0.3961
lstm         test AUC-ROC=0.8528  PR-AUC=0.4744
transformer  test AUC-ROC=0.8502  PR-AUC=0.4711
xgboost_plus_nohist test AUC-ROC=0.8926  PR-AUC=0.5606
xgboost_plus test AUC-ROC=0.8938  PR-AUC=0.5519
=== Error analysis ===
Stratification by history length:
length_bucket  n_cards  n_transactions  fraud_rate  auc_roc  accuracy
          1-2    18549           22240    0.030621 0.892375  0.976304
          3-5     3427           12624    0.049350 0.849855  0.959522
         6-10     1405           10369    0.057672 0.807778  0.951490
        11-20      598            8637    0.061017 0.838683  0.949520
       21-999      239            6312    0.047845 0.824490  0.960551
Error analysis threshold: 0.2649
Error analysis: {
  "false_positive": {
    "n": 1233,
    "mean_n_seen": 6.886455798864558,
    "mean_amount": 188.35336303710938,
    "fraction_synth

In [25]:
!python scripts/phase6b_blend_thresholds.py

Common population: val=69426 rows (3450 fraud), test=60182 rows (2731 fraud)
Blend = 0.85*xgboost_plus + 0.15*lstm (weights chosen on validation PR-AUC=0.6172)
logreg               test AUC-ROC=0.8487  PR-AUC=0.3975  (same 60182 rows for every model)
xgboost              test AUC-ROC=0.8837  PR-AUC=0.5376  (same 60182 rows for every model)
xgboost_plus_nohist  test AUC-ROC=0.8930  PR-AUC=0.5668  (same 60182 rows for every model)
xgboost_plus         test AUC-ROC=0.8956  PR-AUC=0.5580  (same 60182 rows for every model)
lstm                 test AUC-ROC=0.8528  PR-AUC=0.4744  (same 60182 rows for every model)
transformer          test AUC-ROC=0.8502  PR-AUC=0.4711  (same 60182 rows for every model)
blend                test AUC-ROC=0.8957  PR-AUC=0.5604  (same 60182 rows for every model)
Threshold table (thresholds chosen on validation, metrics on test):
              model  strategy  threshold  test_precision  test_recall  test_f1
             logreg        f1     0.8427          0.3799

In [26]:
!python scripts/phase7_visualize.py

Plot 1: confidence vs N
Plot 2: prediction traces
Plot 3: quadrant scatter
Plot 4: attention heatmap
Plot 5: benchmark comparison
Plot 6: confusion matrix
Plot 7 (bonus): embedding t-SNE
All plots saved to /kaggle/working/repo/reports/figures


In [27]:
!cd /kaggle/working/repo && zip -rq /kaggle/working/reports.zip reports checkpoints && ls -lh /kaggle/working/reports.zip


-rw-r--r-- 1 root root 5.9M Nov 18 08:48 /kaggle/working/reports.zip


In [28]:
import numpy as np, pandas as pd
from sklearn.metrics import roc_auc_score, average_precision_score

d = pd.read_parquet("/kaggle/working/repo/data/processed/common_predictions.parquet")
t = d[d.split == "test"].reset_index(drop=True)
groups = t.groupby("card_id").indices
cards = np.array(list(groups))
rng = np.random.default_rng(0)
pairs = [("xgboost_plus", "xgboost_plus_nohist"), ("xgboost_plus", "xgboost"),
         ("blend", "xgboost_plus"), ("xgboost_plus", "lstm")]
out = {p: {"auc": [], "pr": []} for p in pairs}
for _ in range(200):
    sample = rng.choice(cards, len(cards), replace=True)
    s = t.iloc[np.concatenate([groups[c] for c in sample])]
    y = s["isFraud"].to_numpy()
    for a, b in pairs:
        out[(a, b)]["auc"].append(roc_auc_score(y, s[a]) - roc_auc_score(y, s[b]))
        out[(a, b)]["pr"].append(average_precision_score(y, s[a]) - average_precision_score(y, s[b]))
for (a, b), v in out.items():
    for m in ("auc", "pr"):
        lo, hi = np.percentile(v[m], [2.5, 97.5])
        print(f"{a} minus {b} {m}: mean {v[m] and np.mean(v[m]):+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")

xgboost_plus minus xgboost_plus_nohist auc: mean +0.0026  95% CI [-0.0006, +0.0059]
xgboost_plus minus xgboost_plus_nohist pr: mean -0.0092  95% CI [-0.0156, -0.0017]
xgboost_plus minus xgboost auc: mean +0.0120  95% CI [+0.0073, +0.0174]
xgboost_plus minus xgboost pr: mean +0.0202  95% CI [+0.0084, +0.0303]
blend minus xgboost_plus auc: mean +0.0001  95% CI [-0.0011, +0.0015]
blend minus xgboost_plus pr: mean +0.0024  95% CI [-0.0002, +0.0059]
xgboost_plus minus lstm auc: mean +0.0427  95% CI [+0.0338, +0.0509]
xgboost_plus minus lstm pr: mean +0.0830  95% CI [+0.0650, +0.0992]
